In [9]:
from pathlib import Path
import sys
import importlib
import polars as pl 

pl.Config.set_tbl_rows(1000)   # allow many rows
pl.Config.set_tbl_cols(100)    # allow many columns
pl.Config.set_tbl_width_chars(200)


current = Path.cwd()

while current.name != "shared-notebooks":
    if current.parent == current:
        raise RuntimeError("Could not locate shared-notebooks directory")
    current = current.parent

utils_path = current / "common_utils" / "python"
if str(utils_path) not in sys.path:
    sys.path.append(str(utils_path))

import load_flight_data
importlib.reload(load_flight_data)

lf = load_flight_data.load_flight_data(file_name = 'flights_canonical_2019.parquet')

RAW_FEATURES = [
    "flight_id",
    "tail_number",
    "reporting_airline",
    "origin",
    "dest",
    "route_key",
    "distance",
    "flight_date",

    # local calendar/time features
    "dep_hour_local",
    "dep_weekday_local",
    "dep_month_local",

    # UTC timestamps for sequencing
    "dep_ts_sched_utc",
    "dep_ts_actual_utc",
    "arr_ts_sched_utc",
    "arr_ts_actual_utc",

    # weather
    "dep_temp_c",
    "dep_wind_speed_m_s",
    "dep_wind_dir_deg",
    "dep_ceiling_height_m",
    "arr_temp_c",
    "arr_wind_speed_m_s",
    "arr_wind_dir_deg",
    "arr_ceiling_height_m",

    # existing same-tail linkage
    "prev_flight_id_same_tail",
    "next_flight_id_same_tail",
    "prev_origin",
    "prev_dest",
    "next_origin",
    "next_dest",
    "prev_arr_ts_actual_utc",
    "next_dep_ts_actual_utc",

    # delays
    "dep_delay",
    "dep_del15",
    "arr_delay",
    "arr_del15",

    # existing lag features
    "prev_arr_delay",
    "prev_dep_delay",
    "next_arr_delay",
    "next_dep_delay",
    "prev_arr_del15",
    "prev_dep_del15",
    "next_dep_del15",
    "next_arr_del15",
    "prev_arr_late_15",
    "prev_dep_late_15",
    "next_arr_late_15",
    "next_dep_late_15",

    # rotation features
    "turnaround_minutes",
    "next_turnaround_minutes",
    "rotation_continuity_flag",
    "next_rotation_continuity_flag",
    "aircraft_leg_number_day",
    "cum_dep_delay_aircraft_day",
    "cum_arr_delay_aircraft_day",

    # status flags
    "is_cancelled",
    "is_diverted",

    # schedule block features
    "crs_elapsed_time",
    "dep_time_blk",
    "arr_time_blk",
]

# ---------------------------
# US Federal Holidays (2019 example)
# ---------------------------
US_HOLIDAYS_2019 = [
    "2019-01-01",  # New Year's Day
    "2019-01-21",  # MLK Day
    "2019-02-18",  # Presidents Day
    "2019-05-27",  # Memorial Day
    "2019-07-04",  # Independence Day
    "2019-09-02",  # Labor Day
    "2019-10-14",  # Columbus Day
    "2019-11-11",  # Veterans Day
    "2019-11-28",  # Thanksgiving
    "2019-12-25",  # Christmas
]

ml_lf = (
    lf
    .select(RAW_FEATURES)
    .filter(
        (pl.col("is_cancelled") == 0) &
        (pl.col("is_diverted") == 0) &
        pl.col("arr_del15").is_not_null() &
        pl.col("tail_number").is_not_null() &
        pl.col("dep_ts_actual_utc").is_not_null() &
        pl.col("arr_ts_actual_utc").is_not_null()
    )
)

lf_features = (
    ml_lf
    .with_columns([

        # ---------------------------
        # TIME OF DAY BUCKETS
        # ---------------------------
        pl.when(pl.col("dep_hour_local") < 6).then(1)   # early morning
        .when(pl.col("dep_hour_local") < 11).then(2)    # morning
        .when(pl.col("dep_hour_local") < 14).then(3)    # midday
        .when(pl.col("dep_hour_local") < 18).then(4)    # afternoon
        .when(pl.col("dep_hour_local") < 21).then(5)    # evening
        .otherwise(6)                                   # night
        .alias("dep_time_bucket"),

        # ---------------------------
        # WEEKEND FLAG
        # ---------------------------
        pl.col("dep_weekday_local").is_in([6, 7]).cast(pl.Int8).alias("is_weekend"),

        # ---------------------------
        # HOLIDAY FLAG
        # ---------------------------
        pl.col("flight_date")
        .cast(pl.Utf8)
        .is_in(US_HOLIDAYS_2019)
        .cast(pl.Int8)
        .alias("is_holiday"),

        # ---------------------------
        # DAYS TO NEAREST HOLIDAY
        # ---------------------------
        pl.min_horizontal([
            *[
                (pl.col("flight_date").cast(pl.Date) - pl.lit(h).str.strptime(pl.Date)).abs().dt.total_days()
                for h in US_HOLIDAYS_2019
            ]
        ]).alias("days_to_nearest_holiday"),

        # ---------------------------
        # ROUTE FREQUENCY (proxy for traffic density)
        # ---------------------------
        pl.count().over("route_key").alias("route_frequency"),

        # ---------------------------
        # AIRPORT TRAFFIC (congestion proxy)
        # ---------------------------
        pl.count().over("origin").alias("origin_flight_volume"),
        pl.count().over("dest").alias("dest_flight_volume"),

        # ---------------------------
        # DELAY PROPAGATION STRENGTH
        # ---------------------------
        (pl.col("prev_arr_delay") > 15).cast(pl.Int8).alias("prev_arr_delayed_flag"),

        (
            pl.col("prev_arr_delay") +
            pl.col("prev_dep_delay")
        ).alias("prev_total_delay"),

        # ---------------------------
        # TURNAROUND PRESSURE
        # ---------------------------
        (
            pl.col("turnaround_minutes") < 60
        ).cast(pl.Int8).alias("tight_turnaround_flag"),

        # ---------------------------
        # RELATIVE POSITION IN DAY
        # ---------------------------
        (
            pl.col("aircraft_leg_number_day") /
            pl.max("aircraft_leg_number_day").over("tail_number")
        ).alias("relative_leg_position"),

    ])
)

usa_2hop_lf = (
    lf_features
    .sort(["tail_number", "dep_ts_actual_utc"])
    .with_columns([
        # 1 hop back
        pl.col("flight_id").shift(1).over("tail_number").alias("prev1_flight_id"),
        pl.col("origin").shift(1).over("tail_number").alias("prev1_origin"),
        pl.col("dest").shift(1).over("tail_number").alias("prev1_dest"),
        pl.col("dep_ts_actual_utc").shift(1).over("tail_number").alias("prev1_dep_ts_utc"),
        pl.col("arr_ts_actual_utc").shift(1).over("tail_number").alias("prev1_arr_ts_utc"),
        pl.col("arr_delay").shift(1).over("tail_number").alias("prev1_arr_delay"),
        pl.col("dep_delay").shift(1).over("tail_number").alias("prev1_dep_delay"),
        pl.col("arr_del15").shift(1).over("tail_number").alias("prev1_arr_del15"),
        pl.col("dep_del15").shift(1).over("tail_number").alias("prev1_dep_del15"),

        # 2 hops back
        pl.col("flight_id").shift(2).over("tail_number").alias("prev2_flight_id"),
        pl.col("origin").shift(2).over("tail_number").alias("prev2_origin"),
        pl.col("dest").shift(2).over("tail_number").alias("prev2_dest"),
        pl.col("dep_ts_actual_utc").shift(2).over("tail_number").alias("prev2_dep_ts_utc"),
        pl.col("arr_ts_actual_utc").shift(2).over("tail_number").alias("prev2_arr_ts_utc"),
        pl.col("arr_delay").shift(2).over("tail_number").alias("prev2_arr_delay"),
        pl.col("dep_delay").shift(2).over("tail_number").alias("prev2_dep_delay"),
        pl.col("arr_del15").shift(2).over("tail_number").alias("prev2_arr_del15"),
        pl.col("dep_del15").shift(2).over("tail_number").alias("prev2_dep_del15"),

        # timing gaps
        (
            pl.col("dep_ts_actual_utc") -
            pl.col("arr_ts_actual_utc").shift(1).over("tail_number")
        ).dt.total_minutes().alias("prev1_turnaround_minutes"),

        (
            pl.col("dep_ts_actual_utc") -
            pl.col("arr_ts_actual_utc").shift(2).over("tail_number")
        ).dt.total_minutes().alias("time_since_prev2_arrival_minutes"),
    ])
    .filter(
        pl.col("prev1_flight_id").is_not_null() &
        pl.col("prev2_flight_id").is_not_null() &
        pl.col("prev1_turnaround_minutes").is_not_null() &
        pl.col("time_since_prev2_arrival_minutes").is_not_null() &
        pl.col("prev1_turnaround_minutes").is_between(0, 12 * 60) &
        pl.col("time_since_prev2_arrival_minutes").is_between(0, 24 * 60)
    )
)

flights = usa_2hop_lf.collect()
print(flights.height)
print(flights.head())


/tmp/ipykernel_37183/311337846.py:178: DeprecationWarning: `pl.count()` is deprecated. Please use `pl.len()` instead.
(Deprecated in version 0.20.5)
  pl.count().over("route_key").alias("route_frequency"),
/tmp/ipykernel_37183/311337846.py:183: DeprecationWarning: `pl.count()` is deprecated. Please use `pl.len()` instead.
(Deprecated in version 0.20.5)
  pl.count().over("origin").alias("origin_flight_volume"),
/tmp/ipykernel_37183/311337846.py:184: DeprecationWarning: `pl.count()` is deprecated. Please use `pl.len()` instead.
(Deprecated in version 0.20.5)
  pl.count().over("dest").alias("dest_flight_volume"),


6594553
shape: (5, 90)
┌─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┐
│ fli ┆ tai ┆ rep ┆ ori ┆ des ┆ rou ┆ dis ┆ fli ┆ dep ┆ dep ┆ dep ┆ dep ┆ dep ┆ arr ┆ arr ┆ dep ┆ dep ┆ dep ┆ dep ┆ arr ┆ arr ┆ arr ┆ arr ┆ pre ┆ nex ┆ pre ┆ pre ┆ nex ┆ nex ┆ pre ┆ nex ┆ dep ┆ dep ┆ arr ┆ arr ┆ pre ┆ pre ┆ nex ┆ nex ┆ pre ┆ pre ┆ nex ┆ nex ┆ pre ┆ pre ┆ nex ┆ nex ┆ tur ┆ nex ┆ rot ┆ nex ┆ air ┆ cum ┆ cum ┆ is_ ┆ is_ ┆ crs ┆ dep ┆ arr ┆ dep ┆ is_ ┆ is_ ┆ day ┆ rou ┆ ori ┆ des ┆ pre ┆ pre ┆ tig ┆ rel ┆ pre ┆ pre ┆ p

In [11]:
import importlib
import load_faa_registry

importlib.reload(load_faa_registry)

faa_lf = load_faa_registry.load_faa_registry(
    lazy=True,
    refresh=False,
)

flights_enriched_lf = load_faa_registry.join_faa_registry(
    flights=usa_2hop_lf,
    faa=faa_lf,
    flight_tail_col="tail_number",
)

flights_enriched_lf = load_faa_registry.add_dynamic_aircraft_features(
    flights_enriched_lf,
    flight_date_col="flight_date",
    year_mfr_col="year_mfr",
)

flights_enriched = flights_enriched_lf.collect()

print(
    flights_enriched.select([
        "flight_date",
        "tail_number",
        "year_mfr",
        "flight_year",
        "aircraft_age",
        "age_class",
        "age_band",
        "is_older_aircraft",
        "turnaround_per_aircraft_age",
        "prev_arr_delay_x_aircraft_age",
        "daily_utilization_x_age",
    ]).head(20)
)

Using cached FAA parquet: /home/jon/Documents/grad_school/OR568/project/OR568_ML_Project/shared-notebooks/data/faa_registry_enriched.parquet
shape: (20, 11)
┌─────────────┬─────────────┬──────────┬─────────────┬──────────────┬───────────┬──────────┬───────────────────┬─────────────────────────────┬───────────────────────────────┬─────────────────────────┐
│ flight_date ┆ tail_number ┆ year_mfr ┆ flight_year ┆ aircraft_age ┆ age_class ┆ age_band ┆ is_older_aircraft ┆ turnaround_per_aircraft_age ┆ prev_arr_delay_x_aircraft_age ┆ daily_utilization_x_age │
│ ---         ┆ ---         ┆ ---      ┆ ---         ┆ ---          ┆ ---       ┆ ---      ┆ ---               ┆ ---                         ┆ ---                           ┆ ---                     │
│ str         ┆ str         ┆ i32      ┆ i32         ┆ i32          ┆ str       ┆ str      ┆ i8                ┆ f64                         ┆ f64                           ┆ i64                     │
╞═════════════╪═════════════╪══════════

In [2]:
df = load_faa_registry() 

print(df.head())

HTTPError: 403 Client Error: Forbidden for url: https://registry.faa.gov/database/ReleasableAircraft.zip